# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [2]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [3]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [4]:
import string
from nltk.corpus import stopwords 
from nltk import word_tokenize
from nltk.stem.wordnet import WordNetLemmatizer


def clean (text):
    #lowercase
    text = text.lower()

    #Remove numbers
    text = ''.join([word for word in text if not word.isdigit()])

    #Remove punctuation
    for punctuation in string.punctuation:
        text = text.replace(punctuation, ' ')

    #Tokenize
    text = word_tokenize(text)

    #remove stopwords
    stop_words = set(stopwords.words('english'))
    text = [word for word in text if not word in stop_words]

    #Lemmatize
    lemmatize = WordNetLemmatizer()
    text = [lemmatize.lemmatize(word) for word in text]

    #join back into a string
    text = " ".join(text)  

    return text

    #for punctuation in string.punctuation:
    #    text = text.replace(punctuation, ' ') # Remove Punctuation
    #lowercased = text.lower() # Lower Case
    #tokenized = word_tokenize(lowercased) # Tokenize
    #words_only = [word for word in tokenized if word.isalpha()] # Remove numbers
    #stop_words = set(stopwords.words('english')) # Make stopword list
    #without_stopwords = [word for word in words_only if not word in stop_words] # Remove Stop Words
    #lemma=WordNetLemmatizer() # Initiate Lemmatizer
    #lemmatized = [lemma.lemmatize(word) for word in without_stopwords] # Lemmatize
    #cleaned = ' '.join(lemmatized) # Join back to a string
    #return cleaned

# Apply to all texts
#data['clean_text'] = data.text.apply(clean)

#data.head()


In [5]:
data['clean_text'] = data.text.apply(clean)
data.head()

,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gld cunixb cc columbia edu gary l dare subject...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep vela ac oakland edu cardinal ximenez ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,miner kuhub cc ukans edu subject ancient book ...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep vela ac oakland edu cardinal ximenez ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivov superior carleton ca vladimir zhivov s...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [6]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

data_vectorized = vectorizer.fit_transform(data['clean_text'])

lda_model = LatentDirichletAllocation(n_components=2)

lda_vectors = lda_model.fit_transform(data_vectorized)

In [7]:
data_vectorized.shape

(1199, 17492)

In [8]:
lda_vectors.shape

(1199, 2)

##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [9]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [10]:
print_topics(lda_model, vectorizer)

Topic 0:
[('edu', 1081.9999136663255), ('team', 958.4923386284606), ('game', 951.5051556304251), ('line', 718.3089314133571), ('ca', 678.1705896807282), ('hockey', 651.493716447814), ('subject', 636.0327401951682), ('organization', 613.21702272595), ('player', 529.4645318149715), ('play', 520.6439018344008)]
Topic 1:
[('god', 1525.468767435986), ('edu', 1046.0000863336236), ('one', 841.9131426161068), ('christian', 816.7516761237259), ('would', 808.4796360990679), ('people', 700.6573847168145), ('subject', 670.9672598047797), ('line', 638.6910685865909), ('jesus', 626.4898698743162), ('organization', 558.7829772739973)]


In [16]:
lda_model = LatentDirichletAllocation(
    n_components=2, 
    random_state=0
)

lda_vectors = lda_model.fit_transform(data_vectorized)
print_topics(lda_model, vectorizer )

Topic 0:
[('edu', 1061.6613691370615), ('team', 958.4922643003264), ('game', 952.0862288217544), ('line', 712.0455394237356), ('ca', 679.9314521879397), ('hockey', 651.4932290514962), ('subject', 626.8508937829052), ('organization', 609.3422325797018), ('player', 529.4666567552298), ('play', 521.2147151567863)]
Topic 1:
[('god', 1525.468010845448), ('edu', 1066.3386308628872), ('one', 845.6397826897609), ('christian', 816.6518400562669), ('would', 808.8116596484165), ('people', 701.3641931285476), ('subject', 680.1491062170427), ('line', 644.9544605762126), ('jesus', 626.4913489828859), ('organization', 562.6577674202474)]


## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [17]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [18]:
example_vectorized = vectorizer.transform(example)

lda_vectors = lda_model.transform(example_vectorized)

print("topic 0 :", lda_vectors[0][0])
print("topic 1 :", lda_vectors[0][1])

topic 0 : 0.9557543225330593
topic 1 : 0.04424567746694077


In [15]:
print(lda_vectors)
print(lda_vectors.sum(axis=1))

[[0.93778729 0.01545778 0.01576855 0.01557386 0.01541252]]
[1.]


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!